In [1]:
import instructor
from enum import Enum
from openai import AzureOpenAI
from functools import partial
from response_model import PSOM_total,Neuro_Deficit_Score_Severity,Neuro_Deficit_Type,follow_up_status,post_discharge_rehabilitation, QuestionAnswer
import pandas as pd
from datetime import datetime
from tqdm import tqdm
import os
import re
from openpyxl import load_workbook
from openpyxl import Workbook
from typing import List, Optional, Literal, Type
from pydantic import BaseModel
from collections import Counter

In [2]:
# Define your Azure OpenAI configuration
model = "gpt4o_cursor"
engine = "gpt4o_cursor"
endpoint = "https://gpt4o-xj.openai.azure.com/"

api_version = "2024-02-15-preview"
api_type = "azure"

# key_name = "AZURE_KEY"
api_key = '15f19bd128a54ec7b2a10f46299272ec'

# Create the Azure OpenAI client
client = AzureOpenAI(
    api_version=api_version,
    api_key=api_key,
    azure_endpoint=endpoint,
    azure_deployment=engine
)

# Patch the client with instructor
client = instructor.patch(client)

# Optionally, create a partial function with default parameters
in_client = partial(
    client.chat.completions.create,
    model=model,
    temperature=0,
    max_retries=2  # Retries in case of API issues
)


In [3]:
output_directory = "output/"
note_directory = "notes_test/"
results_file_name = 'results.xlsx'
# sheetname = ['PSOM_total','Neuro_Deficit_Score_Severity','Neuro_Deficit_Type','follow_up_status','post_discharge_rehabilitation']
# entities = [PSOM_total,Neuro_Deficit_Score_Severity,Neuro_Deficit_Type,follow_up_status,post_discharge_rehabilitation]
# sheetname = ['Neuro_Deficit_Type','follow_up_status','post_discharge_rehabilitation']
# entities = [Neuro_Deficit_Type,follow_up_status,post_discharge_rehabilitation]
# sheetname = ['Neuro_Deficit_Score_Severity']
# entities = [Neuro_Deficit_Score_Severity]
# sheetname = ['post_discharge_rehabilitation']
# entities = [post_discharge_rehabilitation]
sheetname = ['follow_up_status']
entities = [follow_up_status]
main_field = 'follow_up_status'

In [4]:
# def generate_ask_ai(entity_model):

#     def ask_ai(notes: str):
#         try:
#             cleaned_notes = notes.strip()
#             prompt_template = f'''
# Extract information from the clinical notes.
# Here is the clinical notes: {cleaned_notes}
#             '''
#             # print(prompt_template)
#             # Making a call to the AI client with the appropriate parameters
#             response = in_client(
#                 response_model=entity_model,
#                 messages=[
#                     {
#                         "role": "system",
#                         "content": "You will help extract entities from clinical protocol documents and answer questions.",
#                     },
#                     {
#                         "role": "user",
#                         "content": prompt_template,
#                     },
#                 ],
#                 # validation_context={"text_chunk": notes},
#             )

#             return response

#         except Exception as e:
#             print(f"Error occurred: {e}")
#             return None
#     return ask_ai

In [5]:
# def ask_ai(question: str, notes: str) -> QuestionAnswer:
#     cleaned_notes = notes.strip()

#     response = in_client(
#         response_model = QuestionAnswer,
#         messages = [
#             {
#                 "role": "system",
#                 "content": "You are a clinical note reviewer to answer questions with correct and exact citations.",
#             },
#             {"role": "user", "content": f"{notes}"},
#             {"role": "user", "content": f"Question: {question}"},
#         ],
#         validation_context={"text_chunk": notes},
#     )
#     return response

# with open(note_directory + "30073-1.txt", 'r') as f:
#     notes = f.read()
# question = "After hospital discharge, did the patient receive any of rehabilitation?"
# response = ask_ai(question, notes)

In [6]:
SYSTEM_PROMPT = """
You are a clinical note reviewer. Given a clinical note and a question, follow these steps:

1. Carefully read the clinical note.
2. Identify any exact quotes (phrases) from the note that support a potential answer.
3. From these quotes, infer a fact that answers the question.
4. Based on the fact, answer the question.
5. Explain your reasoning.

Important:
- Only use direct quotes from the clinical note.
- If there is no evidence in the note, answer Unknown.
"""

def ask_ai(notes: str, schema_class: Type[BaseModel]) -> BaseModel:
    # prompt = schema_class.__doc__

    cleaned_notes = notes.strip()
    response = in_client(
        response_model=schema_class,
        messages=[
            {"role": "system", "content": 
                "You are a clinical information extractor. "
                "Your goal is to fill the fields in the given schema by reading the note and reasoning based on direct quotes. "
                "You must only rely on exact quotes from the note. Answer the question based on the fact and explain your reasoning. "
            },
            {"role": "user", "content": f"CLINICAL NOTE:\n{cleaned_notes}"},
            # {"role": "user", "content": f"TASK:\n{prompt.strip()}"},
        ],
        validation_context={"text_chunk": notes},
    )
    return response

In [7]:
n_repeats = 3  

file_path = output_directory + results_file_name
if not os.path.exists(file_path):
    wb = Workbook()
    wb.save(file_path)
for i, Entity in enumerate(entities):
    results = []
    for filename in tqdm(os.listdir(note_directory)):
        # print(filename)
        if filename.startswith('.'):
            continue
        with open(note_directory + filename, 'r') as f:
            notes = f.read()
        # ask_ai = generate_ask_ai(Entity)
        # single call
        # response = ask_ai(notes, Entity)
        # entity_dict = response.dict()
        # multiple calls
        responses = []
        for _ in range(n_repeats):
            try:
                response = ask_ai(notes, Entity)
                responses.append(response)
            except Exception as e:
                print(f"Error occurred during API call: {e}")
                # Optionally, append a default or empty response to maintain consistency
                responses.append(Entity())
        entity_dicts = [r.dict() for r in responses]
        # print(entity_dict)

        # deal with Enums
        for ed in entity_dicts:
            for key, value in ed.items():
                # Check if value is a non-empty list and contains Enums
                if isinstance(value, list) and value and isinstance(value[0], Enum):
                    # If the list is non-empty and contains Enums, extract the string representation
                    ed[key] = [v.value for v in value]
                    # If the value itself is an Enum, get its string representation
                elif isinstance(value, Enum):
                    ed[key] = value.value

        # choose the most common answer and calculate consistency
        all_answers = [ed.get(main_field, None) for ed in entity_dicts]
        answer_counter = Counter(all_answers)

        most_common_answer, count = answer_counter.most_common(1)[0]
        consistency_score = round(count / n_repeats, 2)

        final_output = entity_dicts[0].copy()

        # 取最常见的作为最终写入值
        for ed in entity_dicts:
            if ed.get(main_field, None) == most_common_answer:
                final_output = ed.copy()
                break
        newline = {'filename':filename}
        newline.update(final_output)
        newline.update({
            "answer_consistency": consistency_score,
            "all_answers": all_answers
        })
        results.append(newline)

    df = pd.DataFrame(results)

    with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
            # Write DataFrame to a new sheet
            df.to_excel(writer, sheet_name=sheetname[i])
    
# df.to_csv(output_directory + results_file_name, index=False)

  0%|          | 0/13 [00:00<?, ?it/s]

/var/folders/vh/jcnnz97d2bl8hm9yqsltgx2h0000gn/T/ipykernel_19959/988395535.py:29: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  entity_dicts = [r.dict() for r in responses]
100%|██████████| 13/13 [01:54<00:00,  8.82s/it]
